# UTTA-Med Gate B-FULL — source-only ResNet-18 (Colab GPU)

**Runtime → Change runtime type → T4 GPU** before running.

This trains the frozen source checkpoint on Camelyon17-WILDS:
- train = hospitals `{0, 3, 4}`
- early-stop = OOD val hospital `{1}`
- test hospital `{2}` is evaluated **once**, after the checkpoint is frozen

Do not use the 20-batch smoke checkpoint. Do not run Tent until this notebook finishes.

In [ ]:
import torch, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime → Change runtime type → T4 GPU'

In [ ]:
%pip install -q datasets pyarrow scikit-learn tqdm pillow pyyaml

## Optional: save checkpoints to Google Drive

In [ ]:
from pathlib import Path
SAVE_DIR = Path('/content/utta_med_runs')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = Path('/content/drive/MyDrive/UTTA-Med/runs')
except Exception as e:
    print('Drive not mounted, saving locally:', e)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('SAVE_DIR', SAVE_DIR)

In [ ]:
import json, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm
from datasets import load_dataset

SEED = 42
BATCH = 128
EPOCHS = 20
LR = 1e-4
WD = 1e-4
PATIENCE = 5
NUM_WORKERS = 2
HF_ID = 'wltjr1007/Camelyon17-WILDS'

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda')
print('device', torch.cuda.get_device_name(0))

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def _center_id(v):
    if hasattr(v, 'item'):
        v = v.item()
    return int(v)

class HFCamelyon(Dataset):
    """HuggingFace Camelyon17 filtered by center. validation split is MIXED."""
    def __init__(self, hf_split, centers, transform):
        self.transform = transform
        centers = set(int(c) for c in centers)
        keep = [i for i, row in enumerate(hf_split['center']) if _center_id(row) in centers]
        self.ds = hf_split.select(keep) if keep and len(keep) < len(hf_split) else hf_split
        if keep and len(keep) == len(hf_split):
            self.ds = hf_split
        print(f'  kept {len(self.ds)} / {len(hf_split)}  centers={sorted(centers)}')

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        row = self.ds[i]
        img = row['image']
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img)).convert('RGB')
        else:
            img = img.convert('RGB')
        y = float(row['label'] if not hasattr(row['label'], 'item') else int(row['label']))
        return self.transform(img), torch.tensor([y], dtype=torch.float32)

print('Loading HuggingFace', HF_ID, '(~10.7 GB first time)')
raw = load_dataset(HF_ID)
print({k: len(v) for k, v in raw.items()})

train_set = HFCamelyon(raw['train'], {0, 3, 4}, train_tf)
id_val_set = HFCamelyon(raw['validation'], {0, 3, 4}, eval_tf)
ood_val_set = HFCamelyon(raw['validation'], {1}, eval_tf)
test_set = HFCamelyon(raw['test'], {2}, eval_tf)
print('sizes', len(train_set), len(id_val_set), len(ood_val_set), len(test_set))

In [ ]:
def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=True, drop_last=False)

train_loader = make_loader(train_set, True)
id_loader = make_loader(id_val_set, False)
ood_loader = make_loader(ood_val_set, False)
test_loader = make_loader(test_set, False)

class ResNet18Classifier(nn.Module):
    def __init__(self, dropout_p=0.5):
        super().__init__()
        bb = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        dim = bb.fc.in_features
        bb.fc = nn.Identity()
        self.backbone = bb
        self.classifier = nn.Sequential(
            nn.Linear(dim, dim), nn.ReLU(inplace=True),
            nn.Dropout(dropout_p), nn.Linear(dim, 1),
        )
    def forward(self, x):
        return self.classifier(self.backbone(x))

model = ResNet18Classifier().to(device)
print('params', sum(p.numel() for p in model.parameters())/1e6, 'M')

In [ ]:
def ece_score(y, p, n_bins=15):
    y, p = np.asarray(y).ravel(), np.asarray(p).ravel()
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        m = (p > bins[i]) & (p <= bins[i + 1]) if i else (p >= bins[i]) & (p <= bins[i + 1])
        if m.any():
            ece += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(ece)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        p = torch.sigmoid(model(x)).view(-1).cpu().numpy()
        ys.append(y.view(-1).numpy()); ps.append(p)
    y = np.concatenate(ys); p = np.concatenate(ps); pred = (p >= 0.5).astype(int)
    return {
        'n': int(len(y)),
        'accuracy': float(accuracy_score(y, pred)),
        'precision': float(precision_score(y, pred, zero_division=0)),
        'recall': float(recall_score(y, pred, zero_division=0)),
        'sensitivity': float(recall_score(y, pred, zero_division=0)),
        'specificity': float(((y == 0) & (pred == 0)).sum() / max((y == 0).sum(), 1)),
        'f1': float(f1_score(y, pred, zero_division=0)),
        'auroc': float(roc_auc_score(y, p)),
        'auprc': float(average_precision_score(y, p)),
        'ece': ece_score(y, p),
        'brier': float(np.mean((p - y) ** 2)),
    }

opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
crit = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda')
ckpt = SAVE_DIR / 'camelyon17_resnet18_source_s42.pt'
best, stale, history = -1.0, 0, []
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train(); run = 0.0; n = 0
    pbar = tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}')
    for x, y in pbar:
        x, y = x.to(device, non_blocking=True), y.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        run += loss.item() * x.size(0); n += x.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    val = evaluate(model, ood_loader)
    history.append({'epoch': epoch, 'train_loss': run / max(n, 1), **{f'val_{k}': v for k, v in val.items()}})
    print(f"epoch {epoch:02d} loss={run/max(n,1):.4f} val_auroc={val['auroc']:.4f} val_f1={val['f1']:.4f} val_ece={val['ece']:.4f}")
    if val['auroc'] > best:
        best, stale = val['auroc'], 0
        torch.save({'model': {k: v.detach().cpu() for k, v in model.state_dict().items()},
                    'epoch': epoch, 'val_auroc': best, 'seed': SEED}, ckpt)
        print('  saved', ckpt)
    else:
        stale += 1
        if stale >= PATIENCE:
            print('early stop'); break

state = torch.load(ckpt, map_location='cpu', weights_only=False)
model.load_state_dict(state['model'])
id_m = evaluate(model, id_loader)
ood_m = evaluate(model, ood_loader)
test_m = evaluate(model, test_loader)  # target labels used ONLY here

summary = {
    'experiment_id': 'camelyon17_resnet18_source_s42',
    'device': torch.cuda.get_device_name(0),
    'amp': True, 'epochs_budget': EPOCHS, 'best_epoch': state['epoch'],
    'seconds': round(time.time() - t0, 1),
    'id_val': id_m, 'val_ood': ood_m, 'test_target': test_m,
    'shift': {
        'auroc_drop': id_m['auroc'] - test_m['auroc'],
        'auprc_drop': id_m['auprc'] - test_m['auprc'],
        'f1_drop': id_m['f1'] - test_m['f1'],
        'acc_drop': id_m['accuracy'] - test_m['accuracy'],
        'ece_increase': test_m['ece'] - id_m['ece'],
        'brier_increase': test_m['brier'] - id_m['brier'],
    },
    'history': history,
}
out = SAVE_DIR / 'camelyon17_resnet18_source_s42.json'
out.write_text(json.dumps(summary, indent=2))
print(json.dumps({k: summary[k] for k in ['id_val','val_ood','test_target','shift']}, indent=2))
print('wrote', out)
print('ckpt', ckpt)